In [ ]:
import getpass
from datetime import datetime

def inizialize_stats():
    stats = {}
    stats["saved"] = 0
    stats["discarded"] = 0
    stats["singleton_forward"] = 0
    stats["singleton_reverse"] = 0
    stats["lengths"] = []
    return stats

def register_couple(stats, id_r1:str, len_init_r1:int, len_fin_r1:int, id_r2:str, len_init_r2:int, len_fin_r2:int, result:str):
    """
    Da chiamare dentro il ciclo principale (per ogni coppia elaborata).
    
    Parametri:
    - stats: il dizionario creato con le informazioni di ogni coppia
    - id_r1, id_r2: stringhe con gli ID delle read forward e reverse
    - len_init_r1, len_init_r2: interi con le lunghezze prima del trimming
    - len_fin_r1, len_fin_r2: interi con le lunghezze dopo il trimming
    - esito: stringa che indica cosa è successo alla coppia. Valori ammessi:
             'conservata', 'scartata', 'singleton_r1', 'singleton_r2'
    """
    # Salva le lunghezze iniziali e finali di entrambe le read
    stats["lengths"].append((id_r1, "R1", len_init_r1, len_fin_r1))
    stats["lengths"].append((id_r2, "R2", len_init_r2, len_fin_r2))
    
    # Aggiorna i contatori in base all'esito deciso dalla lunghezza inserita dall'utente
    if result == "conservata":
        stats["saved"] += 1
    elif result == "scartata":
        stats["discarded"] += 1
    elif result == "singleton_r1":
        stats["singleton_forward"] += 1
        stats["discarded"] += 1  # Il singleton vale sempre come coppia scartata
    elif result == "singleton_r2":
        stats["singleton_reverse"] += 1
        stats["discarded"] += 1


def write_report(user_choices, stats, singleton_num:int = 0): #singleton_num è da decidere se calcolarlo o passarlo alla funzione
    """
    Genera il file .txt finale con tutte le informazioni.
    
    Parametri:
    - singleton_num: numero totale dei singleton memorizzati (0 se non si è scelto di salvarli)
    - user_choices: dizionario con i parametri scelti dall'utente. Esempio:
        {
            "file_r1": "sample_R1.fastq",
            "file_r2": "sample_R2.fastq",
            "q_min": 20,
            "l_min": 50,
            "save_singleton": True
        }
    - stats: il dizionario popolato durante l'esecuzione tramite register_couple()
    """
    #calcolo coppie totali
    total_couple = stats["saved"] + stats["discarded"]

    #generazione del nome del file di report
    username = getpass.getuser()
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = "Report_" + username + "_" + timestamp + ".txt"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("====================================================================\n")
        f.write("                 REPORT DI PAIRED-END TRIMMING                      \n")
        f.write("====================================================================\n\n")
        
        #recap delle scelte utente
        f.write("--- 1. PARAMETRI E SCELTE UTENTE ---\n")
        f.write("File input forward:       %s\n" % user_choices["file_r1"])
        f.write("File input reverse:       %s\n" % user_choices["file_r2"])
        f.write("Qualità minima (Qmin):    %d\n" % user_choices["q_min"])
        f.write("Lunghezza minima (Lmin):  %d\n" % user_choices["l_min"])
        
        save_sing = user_choices["save_singleton"]
        if(save_sing):
            f.write("Salvataggio singleton:    Si\n\n")
        else:
            f.write("Salvataggio singleton:    No\n\n")
        
        #recap numero di coppie e singleton
        f.write("--- 2. STATISTICHE COMPLESSIVE ---\n")
        f.write("Coppie totali analizzate: %d\n" % total_couple)
        f.write("Coppie conservate:        %d\n" % stats["saved"])
        f.write("Coppie scartate:          %d\n" % stats["discarded"])
        
        if save_sing:
            tot_singleton = stats["singleton_forward"] + stats["singleton_reverse"]
            f.write("\nDettaglio Singleton salvati:\n")
            f.write("  - Totale singleton:     %d\n" % tot_singleton)
            f.write("  - Singleton forward:    %d\n" % stats["singleton_forward"])
            f.write("  - Singleton reverse:    %d\n" % stats["singleton_reverse"])
        f.write("\n")
        
        #recap lunghezze per ogni coppia di read
        f.write("--- 3. DETTAGLIO LUNGHEZZE PER SINGOLA READ ---\n")
        f.write(f"{'ID READ':<35} {'MATE':<6} {'LEN INIZIALE':<15} {'LEN FINALE':<15}\n")
        f.write("-" * 75 + "\n")
        
        for id_read, mate, l_init, l_fin in stats["lengths"]:
            f.write(f"{id_read:<35} {mate:<6} {l_init:<15} {l_fin:<15}\n")
            
    print("Report salvato correttamente in: %s" % filename)